In [ ]:
import sys
import pandas as pd
import pickle
import importlib

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../../')
sys.path.insert(0, '../../../../../')

# Add path to the other project src to handle imports with dashes in folder names
sys.path.insert(0, '../../../../../probabilistic_suffix_prediction_U-ED-LSTM/src')

In [ ]:
# log as csv
event_log_path = '../../../../data/data/helpdesk.csv'
event_log_df = pd.read_csv(event_log_path)

In [ ]:
# import petri net, 
petri_net_path = '../../../probabilistic_suffix_prediction_U-ED-LSTM/encoded_data_decision/Helpdesk/prefix_for_replay/petri_net/helpdesk.pkl'
with open(petri_net_path, 'rb') as f:
    net, im, fm = pickle.load(f)

# split csv data
train_csv_path = '../../../probabilistic_suffix_prediction_U-ED-LSTM/encoded_data_decision/Helpdesk/prefix_for_replay/helpdesk_all_5_train.csv'
helpdesk_train_df = pd.read_csv(train_csv_path)

val_csv_path = '../../../probabilistic_suffix_prediction_U-ED-LSTM/encoded_data_decision/Helpdesk/prefix_for_replay/helpdesk_all_5_val.csv'
helpdesk_val_df = pd.read_csv(val_csv_path)

test_csv_path = '../../../probabilistic_suffix_prediction_U-ED-LSTM/encoded_data_decision/Helpdesk/prefix_for_replay/helpdesk_all_5_test.csv'
helpdesk_test_df = pd.read_csv(test_csv_path)

# get case ids
unique_list_train = helpdesk_train_df["CaseID"].dropna().unique().tolist()
unique_list_val = helpdesk_val_df["CaseID"].dropna().unique().tolist()

case_ids = list(dict.fromkeys(unique_list_train + unique_list_val))

In [ ]:
import decision_mining.decision_discovery_default
importlib.reload(decision_mining.decision_discovery_default)
from decision_mining.decision_discovery_default import DecisionDiscovery

dd = DecisionDiscovery(petri_net=(net, im, fm),
                       event_log_df=event_log_df,
                       case_ids=case_ids,
                       case_id_key='CaseID',
                       activity_key='Activity',
                       time_key='CompleteTimestamp')

dynamic_attributes = ['Resource']
# static_attributes = ['VariantIndex', 'seriousness', 'customer', 'product', 'responsible_section', 'seriousness_2', 'service_level', 'service_type', 'support_section', 'workgroup']
static_attributes = []

decision_results = dd.decision_mining_datasets(attributes=dynamic_attributes,
                                               trace_attributes=static_attributes)

In [ ]:
# All possible decision points
print(decision_results.decision_points)

m = decision_results.decision_models["p_12"]
print("Chosen decision place: ", m.place_name)
print("Possible next transitions: ", m.outgoing_transitions)
print("Classification model: ", m.classifier)
print("Classification features: ", m.feature_names)
print("Classification classes: ", m.classes)
print("Decision rules: ", m.rules_by_class)
print(m.guard_by_class['Take in charge ticket'])

print("Class encoding: ", m.classifier.classes_)
print("Distribution of classes: ", m.classifier.tree_.value[0][0])